# Customer Churn Prediction — ML Workflow

This notebook trains and evaluates **Logistic Regression** and **Random Forest** using the same cleaning logic as `src/data_cleaning.py`.

**Run from the project root** (the folder containing `data/`, `src/`, and `outputs/`). If using Jupyter, start Jupyter from that folder so the relative paths work.

Workflow:
1. Load and clean the Telco Customer Churn dataset.
2. Prepare features and split the data.
3. Train and tune two classification models.
4. Compare metrics and confusion matrices.
5. Review Random Forest feature importance.
6. Save the results into `outputs/`.


## 1. Import libraries

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
BASE_DIR = Path.cwd()

# If the notebook is opened from notebooks/, use its parent as the project root.
if not (BASE_DIR / "data" / "Telco-Customer-Churn.csv").exists():
    if (BASE_DIR.parent / "data" / "Telco-Customer-Churn.csv").exists():
        BASE_DIR = BASE_DIR.parent

DATA_PATH = BASE_DIR / "data" / "Telco-Customer-Churn.csv"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project folder:", BASE_DIR)
print("Dataset exists:", DATA_PATH.exists())


## 2. Load and clean the dataset

In [ ]:
def load_and_clean_data(data_path=DATA_PATH):
    # Load the original dataset
    df = pd.read_csv(data_path)
    print("Original dataset shape:", df.shape)

    # Remove duplicate rows and rows without a customer ID
    df = df.drop_duplicates()
    if "customerID" in df.columns:
        df = df.dropna(subset=["customerID"])

    # Convert TotalCharges to numeric and fill missing values
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
    df["TotalCharges"] = df["TotalCharges"].fillna(
        df["MonthlyCharges"] * df["tenure"]
    )

    # Create the same TenureGroup feature used by the Python scripts
    df["TenureGroup"] = pd.cut(
        df["tenure"],
        bins=[-1, 12, 24, 48, 72],
        labels=["0-12 Months", "13-24 Months", "25-48 Months", "49-72 Months"]
    )

    print("Cleaned dataset shape:", df.shape)
    missing = df.isnull().sum()
    print("\nColumns with missing values:")
    display(missing[missing > 0].to_frame("Missing values"))

    return df

df = load_and_clean_data()
display(df.head())


## 3. Explore the target variable

In [ ]:
churn_counts = df["Churn"].value_counts().reindex(["No", "Yes"])
display(churn_counts.to_frame("Customers"))

churn_counts.plot(kind="bar", rot=0)
plt.title("Customer Churn Counts")
plt.xlabel("Churn")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.show()

print("Churn percentage:")
display((df["Churn"].value_counts(normalize=True) * 100).round(2).to_frame("Percent"))


## 4. Prepare features and split the data

In [ ]:
# Remove customerID because it is an identifier, not a useful prediction feature.
# The existing training scripts also exclude gender.
X = df.drop(columns=["customerID", "Churn", "gender"], errors="ignore")
y = df["Churn"].map({"No": 0, "Yes": 1})

if y.isna().any():
    raise ValueError("Churn contains missing or unexpected values.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

categorical_cols = X_train.select_dtypes(
    include=["object", "category", "str"]
).columns.tolist()
numerical_cols = X_train.select_dtypes(include=["number"]).columns.tolist()

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)


## 5. Create shared preprocessing

In [ ]:
def make_preprocessor():
    numerical_transformer = SimpleImputer(strategy="median")
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])
    return ColumnTransformer(transformers=[
        ("num", numerical_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ])


## 6. Train Logistic Regression

In [ ]:
lr_pipeline = Pipeline(steps=[
    ("preprocessor", make_preprocessor()),
    ("model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
])

lr_params = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__solver": ["liblinear", "lbfgs"],
    "model__class_weight": [None, "balanced"]
}

lr_search = RandomizedSearchCV(
    estimator=lr_pipeline,
    param_distributions=lr_params,
    n_iter=10,
    scoring="f1",
    cv=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

lr_search.fit(X_train, y_train)
lr_model = lr_search.best_estimator_
lr_prob = lr_model.predict_proba(X_test)[:, 1]

print("Best Logistic Regression parameters:", lr_search.best_params_)
print("Best cross-validation F1:", round(lr_search.best_score_, 4))


## 7. Train Random Forest

In [ ]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", make_preprocessor()),
    ("model", RandomForestClassifier(random_state=RANDOM_STATE))
])

rf_params = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2"],
    "model__class_weight": [None, "balanced", "balanced_subsample"]
}

rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_params,
    n_iter=20,
    scoring="f1",
    cv=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

rf_search.fit(X_train, y_train)
rf_model = rf_search.best_estimator_
rf_prob = rf_model.predict_proba(X_test)[:, 1]

print("Best Random Forest parameters:", rf_search.best_params_)
print("Best cross-validation F1:", round(rf_search.best_score_, 4))


## 8. Compare classification thresholds

In [ ]:
thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]

def threshold_table(y_true, probabilities, model_name):
    rows = []
    for threshold in thresholds:
        predictions = (probabilities >= threshold).astype(int)
        rows.append({
            "Model": model_name,
            "Threshold": threshold,
            "Precision": precision_score(y_true, predictions, zero_division=0),
            "Recall": recall_score(y_true, predictions, zero_division=0),
            "F1-score": f1_score(y_true, predictions, zero_division=0)
        })
    return pd.DataFrame(rows)

lr_thresholds = threshold_table(y_test, lr_prob, "Logistic Regression")
rf_thresholds = threshold_table(y_test, rf_prob, "Random Forest")
threshold_comparison = pd.concat([lr_thresholds, rf_thresholds], ignore_index=True)

display(threshold_comparison.round(4))

lr_threshold = float(lr_thresholds.loc[lr_thresholds["F1-score"].idxmax(), "Threshold"])
rf_threshold = float(rf_thresholds.loc[rf_thresholds["F1-score"].idxmax(), "Threshold"])

print("Selected Logistic Regression threshold:", lr_threshold)
print("Selected Random Forest threshold:", rf_threshold)

threshold_comparison.to_csv(OUTPUT_DIR / "ml_threshold_comparison.csv", index=False)


## 9. Evaluate both models

In [ ]:
def evaluate_model(name, model, probabilities, threshold, best_cv_f1):
    predictions = (probabilities >= threshold).astype(int)
    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Churn Precision": precision_score(y_test, predictions, zero_division=0),
        "Churn Recall": recall_score(y_test, predictions, zero_division=0),
        "Churn F1-score": f1_score(y_test, predictions, zero_division=0),
        "Best Cross-Validation F1-score": best_cv_f1,
        "Selected Threshold": threshold
    }
    print(f"\n===== {name} =====")
    print(classification_report(
        y_test, predictions, labels=[0, 1],
        target_names=["No churn", "Churn"], zero_division=0
    ))
    print("Confusion matrix (rows = actual, columns = predicted):")
    print(confusion_matrix(y_test, predictions, labels=[0, 1]))
    return metrics, predictions

lr_metrics, lr_pred = evaluate_model(
    "Logistic Regression", lr_model, lr_prob, lr_threshold, lr_search.best_score_
)
rf_metrics, rf_pred = evaluate_model(
    "Random Forest", rf_model, rf_prob, rf_threshold, rf_search.best_score_
)

metrics_df = pd.DataFrame([lr_metrics, rf_metrics])
display(metrics_df.round(4))
metrics_df.to_csv(OUTPUT_DIR / "ml_model_comparison.csv", index=False)


## 10. Visual comparison

In [ ]:
metrics_to_plot = ["Accuracy", "Churn Precision", "Churn Recall", "Churn F1-score"]
metrics_df.set_index("Model")[metrics_to_plot].plot(kind="bar", figsize=(10, 5))
plt.title("Logistic Regression vs Random Forest")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, lr_pred, labels=[0, 1], display_labels=["No churn", "Churn"],
    ax=axes[0], colorbar=False
)
axes[0].set_title("Logistic Regression")
ConfusionMatrixDisplay.from_predictions(
    y_test, rf_pred, labels=[0, 1], display_labels=["No churn", "Churn"],
    ax=axes[1], colorbar=False
)
axes[1].set_title("Random Forest")
plt.tight_layout()
plt.show()


## 11. Random Forest feature importance

In [ ]:
feature_names = rf_model.named_steps["preprocessor"].get_feature_names_out()
importance_values = rf_model.named_steps["model"].feature_importances_

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance_values
}).sort_values("Importance", ascending=False)

display(importance_df.head(15).round(5))
importance_df.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)

top_features = importance_df.head(15).sort_values("Importance")
plt.figure(figsize=(9, 6))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.title("Top 15 Random Forest Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


## 12. Findings and business interpretation

In [ ]:
best_row = metrics_df.loc[metrics_df["Churn F1-score"].idxmax()]
print(
    f"Based on test-set F1-score, {best_row['Model']} performed best "
    f"({best_row['Churn F1-score']:.3f})."
)

print("\nTop features from this model run:")
display(importance_df.head(10).round(5))

print(
    "\nInterpretation note: feature importance shows which variables the Random Forest "
    "used most for its predictions. It does not prove that a feature causes churn. "
    "Use the EDA charts and business context alongside these values."
)


## 13. Save final outputs

The notebook saves:
- `outputs/ml_model_comparison.csv`
- `outputs/ml_threshold_comparison.csv`
- `outputs/feature_importance.csv`

**Important:** These metrics may differ slightly from older CSV files if the code, parameter search, package versions, or selected thresholds differ. Use the results produced by this notebook for the report, and do not mix metrics from different runs.
